Task: Parse the stringified JSON array payload into proper Spark array/struct types and explode it to flatten multi-item orders into individual row-level line items.

In [0]:
items_df = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .load("/Volumes/sql_problems/default/my_volume/day12_items.json")
)

items_df.printSchema()
display(items_df)

In [0]:
from pyspark.sql.functions import *

exploded_df = items_df.withColumn(\
    "items",explode(col("items"))
    )
parsed_df = exploded_df.withColumn("item_name",(col("items.item")))\
        .withColumn("price",(col("items.price")))\
        .drop(col("items"))
display(parsed_df)

In [0]:
exploded_df.printSchema()
parsed_df.printSchema()

In [0]:
items_df.createOrReplaceTempView("items_table")

In [0]:
%sql
SELECT
    order_id,
    customer_id,
    order_date,
    item_details.item AS item_name,
    item_details.price AS price
FROM items_table,
LATERAL VIEW EXPLODE(items) e AS item_details

In [0]:
%sql
describe items_table